# 08 · Agent tools (step 10)

The four tools the agent of step 11 will call, run by hand on the real tables. The output below is exactly what the
model will read (JSON). Check: does each answer carry what a historian needs, is anything misleading, is it too long?

* `expand_concept(terms)`: dictionary lookup of one or more words → lemmas, historical spellings, categories, requirement tags with counts
* `search_ads(query, mode, filters, k)`: semantic or keyword search, short hits
* `get_ad(ad_ids)`: full records
* `aggregate(measure, group_by, filters, subset, dimension)`: count / share / pay over **countable** ads

In [ ]:
import json
from hisrag.agent.tools import Tools

tools = Tools()


def run(name: str, **args) -> dict:
    """Call a tool like the agent will, print its JSON and its size."""
    r = tools.call(name, args)
    s = json.dumps(r, ensure_ascii=False, indent=1)
    print(f"{name}({json.dumps(args, ensure_ascii=False)})  →  {len(s)} chars, ~{len(s) // 4} tokens\n{s}")
    return r


specs = json.dumps(tools.specs(), ensure_ascii=False)
print(f"tool specs: {len(specs)} chars, ~{len(specs) // 4} tokens in every request")

## expand_concept

In [ ]:
run("expand_concept", terms=["Köchin", "Gouvernante"]);

In [ ]:
run("expand_concept", terms=["Böhmisch", "Hauslehrer"]);

## search_ads

In [ ]:
r = run("search_ads", query="Welche Sprachkenntnisse wurden von Gouvernanten verlangt?", k=5)

In [ ]:
run("search_ads", query='Gouvernante französisch', mode="keyword", k=3,
    filters={"labels": ["job_search"], "year_to": 1869});

## get_ad

In [ ]:
run("get_ad", ad_ids=[h["ad_id"] for h in r["results"][:2]]);

## aggregate

In [ ]:
run("aggregate", measure="count", group_by="decade", filters={"labels": ["job_offer"]});

In [ ]:
lemmas = [p["lemma"] for p in tools.call("expand_concept", {"terms": ["Lehrer"]})["results"][0]["positions"][:3]]
print(lemmas)
run("aggregate", measure="share", group_by="decade",
    filters={"labels": ["job_offer"], "position_lemmas": lemmas},
    subset={"requirement_tags": ["sprachkenntnisse:Böhmisch"]});

In [ ]:
run("aggregate", measure="count", group_by="requirement_value", dimension="sprachkenntnisse",
    filters={"labels": ["job_offer"]});

In [ ]:
cooks = [p["lemma"] for p in tools.call("expand_concept", {"terms": ["Köchin"]})["results"][0]["positions"][:1]]
print(cooks)
run("aggregate", measure="pay", group_by="decade", filters={"position_lemmas": cooks});

In [ ]:
run("aggregate", measure="share", group_by="decade", filters={"labels": ["job_search"]},
    subset={"keyword": "Gouvernante*"});

In [ ]:
run("aggregate", measure="count", group_by="benefit", filters={"labels": ["job_offer"], "year_to": 1879});

## Errors come back to the model

In [ ]:
run("aggregate", group_by="requirement_value");
run("search_ads", query="x", k=100);